# 05 · Ask several questions of one cube

## Context

A single environmental record can support several legitimate summaries. A
researcher may care about typical conditions, variability, departures from a
baseline, comparable scales, or a matrix suitable for modeling.

## Question

How does the choice of verb change both the scientific meaning and the shape of
the result?

## Analysis story

We will hold the evidence constant and vary only the analytical question. A
small set of parallel pipes makes those choices easy to compare.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

from cubedynamics import pipe, verbs as v

# Build one cube with temporal trend, seasonality, spatial structure, and noise.
# Every verb below therefore starts from exactly the same evidence.
rng = np.random.default_rng(8)
time = pd.date_range("2023-01-01", periods=24, freq="MS")
y = np.arange(5)
x = np.arange(6)
trend = np.linspace(0, 2, time.size)[:, None, None]
season = 2 * np.sin(2 * np.pi * np.arange(time.size)[:, None, None] / 12)
landscape = np.linspace(-1, 1, y.size)[None, :, None] + np.linspace(0, 1, x.size)[None, None, :]
cube = xr.DataArray(
    10 + trend + season + landscape + rng.normal(0, 0.3, (24, 5, 6)),
    dims=("time", "y", "x"),
    coords={"time": time, "y": y, "x": x},
    name="signal",
)
cube

## Pipes · One input, six analytical questions

Each expression is intentionally short. Reducers remove a dimension,
transforms preserve it, and `v.flatten_space` prepares a matrix for modeling.

In [ ]:
# What is typical at each location?
mean_map = (
    pipe(cube)
    | v.mean(dim="time", keep_dim=False)
).unwrap()

# Where does the signal vary most?
variance_map = (
    pipe(cube)
    | v.variance(dim="time", keep_dim=False)
).unwrap()

# How far is each value from its local baseline?
anomaly = (
    pipe(cube)
    | v.anomaly(dim="time")
).unwrap()

# How unusual is each value on a common scale?
zscore = (
    pipe(cube)
    | v.zscore(dim="time")
).unwrap()

# How can a project rule limit extreme standardized values?
clipped = (
    pipe(zscore)
    | v.apply(lambda value: value.clip(min=-1, max=1))
).unwrap()

# How can the anomaly become a time × feature matrix?
flat = (
    pipe(anomaly)
    | v.flatten_space(new_dim="pixel")
).unwrap()

assert flat.dims == ("time", "pixel")

## Figure · Compare the consequences

The six panels make shape and meaning visible. Read each title as the question
answered by the pipe above it.

In [ ]:
import matplotlib.pyplot as plt

# Each panel is labeled with the verb and the question its output can answer.
fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
mean_map.plot(ax=axes[0, 0], cmap="viridis")
axes[0, 0].set_title("v.mean: typical spatial pattern")
variance_map.plot(ax=axes[0, 1], cmap="magma")
axes[0, 1].set_title("v.variance: variable locations")
anomaly.isel(time=-1).plot(ax=axes[0, 2], cmap="RdBu_r", center=0)
axes[0, 2].set_title("v.anomaly: departure from normal")
zscore.sel(y=2, x=3).plot(ax=axes[1, 0], color="#3f6f72")
axes[1, 0].axhline(0, color="0.45", linewidth=0.8)
axes[1, 0].set_title("v.zscore: comparable units")
clipped.isel(time=-1).plot(ax=axes[1, 1], cmap="RdBu_r", vmin=-1, vmax=1)
axes[1, 1].set_title("v.apply: project function")
axes[1, 2].imshow(flat.values.T, aspect="auto", cmap="RdBu_r")
axes[1, 2].set(title="v.flatten_space: time × pixel", xlabel="time index", ylabel="pixel")
plt.show()

## What the figure tells us

No verb is universally “best.” Means and variance summarize across time;
anomalies and z-scores preserve timing; clipping expresses a project decision;
flattening changes representation for a downstream model. The pipe keeps each
choice visible and reviewable.

## Try the next variation

Add one new pipe with `v.sum`, or change the reduction dimension from `time` to
space. State the question first, then choose the verb.